# 02. Data Acquisition: Flight Price Extraction on Serpapi

This notebook serves as the initial data collection pipeline. It retrieves one-way flight fare data via SerpApi (Google Flights) connector.

## Objectives
- Extraction: Fetch real-time pricing data for targeted route pairs originating from/returning to Singapore (SIN).
- Parameters: Execute searches on predefined travel dates

### Environment Setup

In [1]:
import serpapi
import pandas as pd
import json
import time
import os
from datetime import datetime 
from serpapi import GoogleSearch
from dotenv import load_dotenv

In [2]:
# Load API from .env
load_dotenv()
api = os.environ.get("SERPAPI_KEY")

### Flight Price Extraction

In [3]:
# Target routes
destinations = ["BKK", "NRT", "LHR", "MEL"]

# Outbound travel dates
outbound_dates = ["2026-05-30", "2026-06-11", "2026-08-03", "2026-07-15"]

# Search date / extracted date
extracted_date_str = datetime.now().strftime("%Y-%m-%d")

# Loop through travel dates independently 
for outbound_date in outbound_dates:
    date_batch_responses = {}
    filename = f"serpapi_response_{outbound_date}_search{extracted_date_str}.json"

    # Loop through destinations 
    for dest in destinations:

        #Building vice-versa bidirectional segments
        sectors = [
            {"from": "SIN", "to": dest, "direction": "outbound"},
            {"from": dest, "to": "SIN", "direction": "inbound"}
        ]

        # Execute outbound followed by return leg
        for sector in sectors:
            dep = sector["from"]
            arr = sector["to"]
            direction_flag = sector["direction"]
            route_label = f"{dep}-{arr}_{outbound_date}"

            print(f"Running request {route_label} | Date: {outbound_date} {direction_flag}")

            # Building the parameters
            params = {
                "engine": "google_flights",
                "departure_id": dep,
                "arrival_id": arr,
                "currency": "SGD",
                "type": "2",
                "outbound_date": outbound_date,
                "gl":"sg",
                "hl":"en",
                "sort_by":"2",
                "stops":"1",
                "api_key":api
            }

            try:
                search = GoogleSearch(params)
                results = search.get_dict()

                if "error" in results: 
                    print(f"API Server Error on {route_label}: {results['error']}")
                    date_batch_responses[route_label] = {"status": "error", "message": results["error"]}
                else: 
                    best_count = len(results.get("best_flights", []))
                    other_count = len(results.get("other_flights", []))
                    print(f"Success: Parsed {best_count + other_count} non-stop itineraries into memory.")

                    # Store results under its route market index key
                    date_batch_responses[route_label] = results

            except Exception as e: 
                print(f"Connection breakdown for {route_label}: {e}")
                date_batch_responses[route_label] = {"status": "failed", "message": str(e)}

            # 2-second delay barrier to regulate request volume saefty
            time.sleep(2)

    # Exporting files to JSON
    try:
        with open(filename, "w", encoding="utf-8") as json_file:
            json.dump(date_batch_responses, json_file, indent=4, ensure_ascii=False)
        print(f"Batch export finalized completely for file: '{filename}'")
    except Exception as file_err:
        print(f"Critical file saving failure on: {file_err}")

Running request SIN-BKK_2026-05-30 | Date: 2026-05-30 outbound
Success: Parsed 16 non-stop itineraries into memory.
Running request BKK-SIN_2026-05-30 | Date: 2026-05-30 inbound
Success: Parsed 13 non-stop itineraries into memory.
Running request SIN-NRT_2026-05-30 | Date: 2026-05-30 outbound
Success: Parsed 6 non-stop itineraries into memory.
Running request NRT-SIN_2026-05-30 | Date: 2026-05-30 inbound
Success: Parsed 5 non-stop itineraries into memory.
Running request SIN-LHR_2026-05-30 | Date: 2026-05-30 outbound
Success: Parsed 7 non-stop itineraries into memory.
Running request LHR-SIN_2026-05-30 | Date: 2026-05-30 inbound
Success: Parsed 7 non-stop itineraries into memory.
Running request SIN-MEL_2026-05-30 | Date: 2026-05-30 outbound
Success: Parsed 11 non-stop itineraries into memory.
Running request MEL-SIN_2026-05-30 | Date: 2026-05-30 inbound
Success: Parsed 11 non-stop itineraries into memory.
Batch export finalized completely for file: 'serpapi_response_2026-05-30_search2